In [1]:
import pandas as pd

# 1) Charger (adapte le nom si ton fichier s'appelle autrement)
gepu = pd.read_excel("Global_Policy_Uncertainty_Data.xlsx")

# 2) Convertir les colonnes numériques avec virgules
for c in ["GEPU_current", "GEPU_ppp"]:
    gepu[c] = pd.to_numeric(gepu[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

# 3) Construire une date mensuelle (1er du mois)
gepu["Date"] = pd.to_datetime(
    gepu["Year"].astype(int).astype(str) + "-" +
    gepu["Month"].astype(int).astype(str).str.zfill(2) + "-01",
    errors="coerce"
)

# 4) Trier + garder propre
gepu = gepu.sort_values("Date")[["Date", "GEPU_current", "GEPU_ppp"]].dropna(subset=["Date"]).set_index("Date")

C:\Users\lilia\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [2]:
# 5) Prendre GEPU_current + créer le lag (t-1)
gepu_m = gepu[["GEPU_current"]].copy()
gepu_m["GEPU_lag1"] = gepu_m["GEPU_current"].shift(1)

# 6) Charger la base préprocess (Fama + portefeuilles)
df = pd.read_excel("data_FF_PF.xlsx")

# 7) Date en datetime + index mensuel (1er du mois)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

# ✅ alignement au 1er du mois (version qui marche partout)
df.index = df.index.to_period("M").to_timestamp(how="start")

# 9) Merger df + GEPU (sur l'index Date)
df_gepu = df.merge(gepu_m, left_index=True, right_index=True, how="left")

# 10) Garder uniquement la période où GEPU_lag1 existe (1997+)
df_gepu_model = df_gepu.dropna(subset=["GEPU_lag1"]).copy()

print("Période utilisable :", df_gepu_model.index.min(), "→", df_gepu_model.index.max())
df_gepu_model[["GEPU_current", "GEPU_lag1"]].head()

Période utilisable : 1997-02-01 00:00:00 → 2025-10-01 00:00:00


,GEPU_current,GEPU_lag1
Date,,
1997-02-01,76.076043,75.741272
1997-03-01,66.713953,76.076043
1997-04-01,71.674360,66.713953
1997-05-01,75.498983,71.674360
1997-06-01,79.464737,75.498983


In [3]:
df_gepu_model.head()

,SMALL_LoINV,BIG_LoINV,Mkt_RF,SMB,HML,RMW,CMA,RF,SMALL_LoINV_RF,BIG_LoINV_RF,GEPU_current,GEPU_lag1
Date,,,,,,,,,,,,
1997-02-01,0.0437,0.9163,-0.48,-2.54,5.61,0.72,3.47,0.39,-0.3463,0.5263,76.076043,75.741272
1997-03-01,-4.4646,-3.8884,-5.02,-0.41,3.39,0.39,1.53,0.43,-4.8946,-4.3184,66.713953,76.076043
1997-04-01,-1.8233,3.5301,4.04,-5.70,0.00,3.36,-1.02,0.43,-2.2533,3.1001,71.674360,66.713953
1997-05-01,10.1108,5.7632,6.73,4.68,-4.11,-1.04,-2.76,0.49,9.6208,5.2732,75.498983,71.674360
1997-06-01,4.8933,4.2993,4.10,1.17,1.44,0.56,0.45,0.37,4.5233,3.9293,79.464737,75.498983


In [4]:
# Export en Excel (simple)
df_gepu_model.to_excel("data_FF_PF_GEPU.xlsx", index=True)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# 1️⃣ Volatilité réalisée (historique)
realized_vol_big = big_garch_result.conditional_volatility

# 2️⃣ Volatilité future prévue
forecast_horizon = len(big_volatility_future)

last_date = df.index[-1]

# Tes données Fama-French sont mensuelles
future_dates = pd.date_range(start=last_date, periods=forecast_horizon+1, freq='M')[1:]

forecast_big_series = pd.Series(big_volatility_future.values, index=future_dates)

# 3️⃣ Graphique propre : bleu s'arrête, orange continue
plt.figure(figsize=(12,6))

plt.plot(df.index, realized_vol_big,
         label="Volatilité réalisée (BIG)", linewidth=1.5)

plt.plot(forecast_big_series.index, forecast_big_series.values,
         label="Prévision GARCH (BIG)", linestyle="--", linewidth=2)

plt.axvline(x=last_date, linestyle=":", linewidth=1)
plt.text(last_date, plt.ylim()[1]*0.95,
         "Début prévision", rotation=90, va="top")

plt.title("GARCH(1,1) — Prévision de volatilité (BIG)")
plt.xlabel("Date")
plt.ylabel("Volatilité")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'big_garch_result' is not defined